# Solvating an MDMC Universe  

 

For simulating systems in solution, MDMC's inbuilt ``solvate`` method can be used to add solvent molecules of a desired density to your universe.



# Create the Universe
----
See the [_Building a Universe_](https://github.com/MDMCproject/MDMCv0.2_pilot/wiki/Building-a-Universe)  wiki for details on how to create a Universe of your own specifications. For the purposes of this tutorial, we will be solvating a simple universe that contains only 4 Hydrogen molecules:

In [3]:
# Import the Atom, Molecule, and Universe classes
# Import the HarmonicPotential class (needed to create a Bond)
from MDMC.MD.simulation import Universe
from MDMC.MD.structural_units import Atom, Bond, Molecule
from MDMC.MD.interaction_functions import HarmonicPotential

# Initialise a Universe with dimensions in Ang
universe = Universe([10.0, 15.0, 20.0])

# Create a pair of Hydrogen Atoms
H1 = Atom('H')
H2 = H1.copy(position=[1., 1., 1.])

# Initialise a H-H Bond
HH_bond = Bond(H1, H2, function=HarmonicPotential((1., 'Ang'), 
                                                  (100., 'kJ / mol Ang^2')))

# Make a H2 Molecule
H2_mol_1 = Molecule(atoms=[H1, H2], interactions=[HH_bond])

# Create 3 copies of the Molecule at different positions
H2_mol_2 = H2_mol_1.copy(position=[3.5, 3.5, 3.5])
H2_mol_3 = H2_mol_1.copy(position=[6.5, 6.5, 6.5])
H2_mol_4 = H2_mol_1.copy(position=[9.5, 9.5, 9.5])

# Add the 4 Hydrogen Molecules to the Universe
for molecule in [H2_mol_1, H2_mol_2, H2_mol_3, H2_mol_4]:
    universe.add_structural_unit(molecule)

# Solvating the Universe
----
The ```solvate``` method accepts 3 main parameters; ``density``, ``tolerance``, and ``solvent`` (as well as some ``**settings``). An example call to ```solvate``` the Universe created above would therefore be:


In [ ]:
universe.solvate(0.6, tolerance=1, solvent='SPCE')

## Parameters 

Explanations of each parameter passed to ``solvate`` can be found below.


### 1. ``density``
The desired density of the bulk solvent in your universe, in MDMC units of **amu / Ang ^ 3**. See this tutorial **.....LINK TO UNIT CONVERSION WIKI.....** for instructions on how to convert your density into MDMC units.


**Note**: with this parameter you are specifying the **bulk density of the solvent**. If you have any solute molecules already present in your universe, the total density of your universe after solvation will be higher than the desired density you pass to ``solvate`` (plus or minus the tolerance you pass, see below).

In the above example, passing the density as ``0.6`` means that the universe will be solvated with SPCE water with a bulk density of 0.6 amu / Ang ^ 3 (+/- the tolerance). The density of the universe **in total will be greater** than 0.6 amu / Ang ^ 3, because of the 4 Hydrogen molecules.



### 2. ``tolerance``
With this parameter you can specify the percentage tolerance of the ``density`` that you would like to be achieved for the bulk density of the solvent. 

For the above example, passing a density of ``0.6`` and setting ``tolerance=1`` will achieve a bulk solvent density of:

* (0.6 amu / Ang ^ 3)  +/-  1 %
* equivalent to (0.6  +/-  0.006) amu / Ang ^ 3

**Note**: the tolerance has a default value of 1 %. Setting the tolerance to anything lower than increases the risk of ``solvate`` not converging to within the specified tolerance.



### 3. ``solvent``


#### a) Using a solvent with pre-defined coordinates

MDMC has a few in-built solvents that can be used to solvate your Universe. These have pre-defined atomic coordinates and interactions. 

Currently, the in-built solvents you can choose from are:  

* SPCE water

##### Example: solvating with SPCE water

In [ ]:
universe.solvate(0.6, tolerance=1, solvent='SPCE')

#### b) Specifying a StructuralUnit.
You can also create a StructuralUnit (such as an Atom or Molecule) with which you can solvate the universe. This will add the molecule in random positions in the Universe until the density is within the specified tolerance. **While it is good practice to energy minimize and equilibrate all simulations before refinement, it is strongly advised for a** ```Universe``` **solvated in this manner.**

##### Example: solvating with a methanol molecule

Methanol coordinates taken from [Biological Magnetic Resonance Data Bank](http://www.bmrb.wisc.edu/ftp/pub/bmrb/metabolomics/entry_directories/bmse000294/bmse000294.mol)

Bonded interaction parameters are from OPLSAA.  Dihedrals and non-bonded interactions have not been included for this tutorial.


In [6]:
from itertools import combinations

# Methanol requires bond angles to be defined
from MDMC.MD.structural_units import BondAngle

# Define the unique atoms
# The H1 atom will be copied after the bond and bond angles have been defined
HC1 = Atom('H', position=[-0.7006,  0.3636,  0.8900], charge=0.04)
C = Atom('C', position=[-0.3366, -0.1504,  0.0000], charge=0.145)
O = Atom('O', position=[ 1.0849, -0.1713,  0.0000], charge=-0.683)
HO1 = Atom('H', position=[ 1.3606,  0.7699,  0.0000], charge=0.418)

# Create the bonds with harmonic potentials
CH_bond = Bond(C, HC1, function=HarmonicPotential((1.0800, 'Ang'), (284512., 'kJ / mol Ang^2')))
CO_bond = Bond(C, O, function=HarmonicPotential((1.2290, 'Ang'), (476976., 'kJ / mol Ang^2')))
OH_bond = Bond(O, HO1, function=HarmonicPotential((0.9450, 'Ang'), (462750.4, 'kJ / mol Ang^2')))

# Create the H-C-O and H-O-C bond angles
HCO_angle = BondAngle((HC1, C, O),
                      function=HarmonicPotential((115.000, 'deg'), (5.84197, 'kJ / mol deg^2')))
HOC_angle = BondAngle((HO1, O, C),
                      function=HarmonicPotential((113.000, 'deg'), (5.11172, 'kJ / mol deg^2')))

# Duplicate the HC1 atom
HC2 = HC1.copy(position=[-0.7006,  0.3636, -0.8900])

# Create an HCH bond angle
HCH_angle = BondAngle((HC1, C, HC2),
                      function=HarmonicPotential((109.5, 'deg'), (4.81962, 'kJ / mol deg^2')))

# Duplicate the HC1 atom again
# This atom will have all bond (CH_bond) and bond angles (HCO_angle and HCH_angle) defined
HC3 = HC1.copy(position=[-0.7076, -1.1754,  0.0000])

# Create the methanol Molecule
methanol = Molecule(atoms=[HC1, HC2, HC3, C, O, HO1])

# Solvate the universe created above with the methanol StructuralUnit
universe.solvate(0.792, tolerance=1, solvent=methanol)

#### c) Specifying a Configuration.

You can also pass ``solvate`` a Configuration to solvate your universe with.

##### Example: 


In [ ]:
universe.solvate(0.75, tolerance=0.8, solvent=my_config)

### 4. ``**settings``

#### a) ``constraint_algorithm``

You can specify ConstraintAlgorithm which is applied to the Universe. If specifying ``solvent`` as a string representing one of the in-built solvents (i.e. 'SPCE'), then a ``Shake(1e-4, 100)`` constraint algorithm is applied by default.
